In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV, KFold, cross_validate
from sklearn.metrics import make_scorer, r2_score, mean_squared_error, mean_absolute_error

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from scipy.stats import randint, uniform

import os
import joblib

In [2]:
# 1- xgboost
# 2- adaboost
# 3- gradient boosting
# 4- catboost
# 5- lightboost
# 6- random forest
# 7- stacking 01
# 8- stacking 02

In [3]:
train_df = pd.read_csv("../data/train_preprocessed.csv")
test_df = pd.read_csv("../data/test_preprocessed.csv")

target_cols = ["CO2 Emission (Kg/m3)", "Compressive_strength"]

y_train = train_df[target_cols]
X_train = train_df.drop(columns=target_cols)

y_test = test_df[target_cols]
X_test = test_df.drop(columns=target_cols)

X_train.shape, X_test.shape

((513, 17), (129, 17))

# gradient boosting

In [ ]:
gbr = GradientBoostingRegressor(random_state=42)
model = MultiOutputRegressor(gbr)

param_dist = {
    "estimator__n_estimators": randint(100, 600),
    "estimator__learning_rate": uniform(0.01, 0.29),  
    "estimator__subsample": uniform(0.6, 0.4),

    "estimator__max_depth": randint(2, 8),
    "estimator__min_samples_split": randint(2, 20),
    "estimator__min_samples_leaf": randint(1, 20),

    "estimator__max_features": ["sqrt", "log2", None],
}

cv = KFold(n_splits=10, shuffle=True, random_state=42)

random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_dist,
    n_iter=100,
    scoring="r2",
    cv=cv,
    verbose=2,
    random_state=42,
    n_jobs=4,
    error_score="raise"
)

random_search.fit(X_train, y_train)

print("Best Hyperparameters:", random_search.best_params_)
print("Best R2:", random_search.best_score_)

Fitting 10 folds for each of 100 candidates, totalling 1000 fits


In [ ]:
best = random_search.best_params_

base_gbr = GradientBoostingRegressor(random_state=42)
model = MultiOutputRegressor(base_gbr)

grid = {
    "estimator__n_estimators": sorted(set([
        max(50, best["estimator__n_estimators"] - 60),
        max(50, best["estimator__n_estimators"] - 30),
        best["estimator__n_estimators"],
        best["estimator__n_estimators"] + 30,
    ])),

    "estimator__learning_rate": sorted(set([
        max(0.005, best["estimator__learning_rate"] * 0.7),
        max(0.005, best["estimator__learning_rate"] * 0.85),
        best["estimator__learning_rate"],
        min(0.5, best["estimator__learning_rate"] * 1.15),
    ])),

    "estimator__max_depth": sorted(set([
        max(1, best["estimator__max_depth"] - 1),
        best["estimator__max_depth"],
        best["estimator__max_depth"] + 1,
    ])),

    "estimator__min_samples_leaf": sorted(set([
        max(1, best["estimator__min_samples_leaf"] - 1),
        best["estimator__min_samples_leaf"],
        best["estimator__min_samples_leaf"] + 1,
    ])),

    "estimator__min_samples_split": [
        best["estimator__min_samples_split"]
    ],
}

cv = KFold(n_splits=10, shuffle=True, random_state=42)

n_candidates = 1
for v in grid.values():
    n_candidates *= len(v)
print("Grid candidates:", n_candidates, "| Total fits:", n_candidates * cv.get_n_splits())

grid_search = GridSearchCV(
    estimator=model,
    param_grid=grid,
    scoring="r2",
    cv=cv,
    verbose=2,
    n_jobs=4,
    error_score="raise"
)

grid_search.fit(X_train, y_train)

print("Best params (Grid):", grid_search.best_params_)
print("Best R2 (Grid):", grid_search.best_score_)


In [ ]:
best_model = grid_search.best_estimator_

cv = KFold(n_splits=10, shuffle=True, random_state=42)

scoring = {
    "r2": "r2",
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error"
}

cv_res = cross_validate(
    best_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=4,
    return_train_score=False
)

print("CV R2 mean ± std:", cv_res["test_r2"].mean(), "±", cv_res["test_r2"].std())
print("CV RMSE mean ± std:", (-cv_res["test_rmse"]).mean(), "±", (-cv_res["test_rmse"]).std())
print("CV MAE mean ± std:", (-cv_res["test_mae"]).mean(), "±", (-cv_res["test_mae"]).std())


In [ ]:
y_pred = best_model.predict(X_test)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))
mae_test = mean_absolute_error(y_test, y_pred)

print("TEST RMSE:", rmse_test)
print("TEST MAE:", mae_test)

In [ ]:
r2_per_target = r2_score(y_test, y_pred, multioutput="raw_values")
rmse_per_target = np.sqrt(mean_squared_error(y_test, y_pred, multioutput="raw_values"))
mae_per_target = mean_absolute_error(y_test, y_pred, multioutput="raw_values")

for i, col in enumerate(y_test.columns):
    print(f"{col}: R2={r2_per_target[i]:.4f}, RMSE={rmse_per_target[i]:.4f}, MAE={mae_per_target[i]:.4f}")

In [ ]:
model_dir = "../ml_models"
os.makedirs(model_dir, exist_ok=True)

model_path = os.path.join(model_dir, "gradient_boosting_final_model.pkl")
joblib.dump(best_model, model_path)  

print(f"Model saved to: {model_path}")
